# Ensemble stability, failures, nulls and the final test (Prompt #11, Steps 37-45, 49-50, 59-65, 75)

By year, quarter, regime, volatility, spread and session; the alpha decay and the decile shape; where the ensemble fails and whether every model fails together; the null ensembles; the freeze decision and the one-time (second-look) final test.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.ensemble` and `xauusd_quant.research.ensemble_*` and read back from `results/ensemble_research/`. Generate those first:

```
xq ensemble-research --timeframe 5m --report   # the meta walk-forward of every pair
xq ensemble-freeze --timeframe 5m              # pre-registered rule -> ENSEMBLE_SPEC
xq ensemble-finalize --timeframe 5m            # constituents, streaming, benchmark
xq ensemble-final-test --ensemble-spec ENS_REVERSION_5M_H5_V001   # once; logged
```

Every method is scored on the evaluation blocks 2013-2021 and fitted on earlier out-of-sample blocks only (purged by horizon + embargo). Probabilities, expected values and uncertainty proxies only - no signal, threshold, size or PnL.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.ensemble.config import load_ensemble_config

cfg = load_ensemble_config(ROOT / "config" / "ensemble.yaml")

TIMEFRAME = "5m"                      # 5m | 15m - change and re-run
TARGET, HORIZON = "mean_reversion", 5  # any pair of config/ensemble.yaml

out = cfg.results_path / TIMEFRAME
pair_dir = out / TARGET / f"h{HORIZON}"
if not pair_dir.exists():
    print(f"No ensemble research for {TIMEFRAME} {TARGET} h{HORIZON}. Run:  "
          f"xq ensemble-research --timeframe {TIMEFRAME} --report")

def table(name, where=None):
    """A stored table of the pair (or of the timeframe with where='tf'); None if absent."""
    base = out / "tables" if where == "tf" else pair_dir
    p = base / f"{name}.parquet"
    return pl.read_parquet(p) if p.exists() else None

def plot(name, where=None):
    """Show a stored figure of the pair (or of the timeframe with where='tf')."""
    base = out / "plots" if where == "tf" else pair_dir / "plots"
    p = base / f"{name}.png"
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f"no figure {p.name}")

def summary():
    p = pair_dir / "summary.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


## By year / quarter / state

In [ ]:
table('stability').filter(pl.col('series').is_in(['best_individual', 'simple_average'])).sort('grouping', 'period', 'series')

In [ ]:
plot('performance_by_year')
plot('performance_by_state')

## Alpha decay and deciles (Steps 37-38)

In [ ]:
table('alpha_decay')

In [ ]:
plot('ic_decay')
plot('deciles')

## Failure analysis and common-mode failure (Steps 49-50)

In [ ]:
table('failure_analysis').sort('lift', descending=True)

In [ ]:
table('common_mode').sort('lift', descending=True)

In [ ]:
{k: summary().get(k) for k in ('failure_summary', 'common_mode_summary')}

## Null ensembles (Step 75)

In [ ]:
table('null_ensembles')

## The freeze decision (Steps 54, 59-61, 84)

In [ ]:
json.loads((pair_dir / 'freeze_decision.json').read_text(encoding='utf-8'))

## Every timeframe's freeze decisions

In [ ]:
table('freeze_decisions', where='tf')

## The one-time final test (a second look; Steps 62-65)

In [ ]:
p = out / 'final_test' / 'ensemble_summaries.json'
json.loads(p.read_text(encoding='utf-8')) if p.exists() else 'not run'

In [ ]:
plot('overview_methods', where='tf')